<a href="https://colab.research.google.com/github/bhaskarkumar1667/sih_2026/blob/main/cnn_modeltraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -r requirements.txt


  Using cached torch-2.14.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (37 kB)
  Using cached torch_geometric-2.8.0.post1-py3-none-any.whl.metadata (64 kB)
  Using cached matplotlib-3.11.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached scikit_learn-1.9.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached cuda_bindings-13.3.1-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.5 kB)
  Using cached nvidia_cudnn_cu13-9.24.0.43-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cublas-13.1.1.3-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cufft-12.0.0.61-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cusolver-12.0.4.66-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cusparse-12.6.3.3-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.8 kB)

In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import networkx as nx
import numpy as np
import pandas as pd
import random
from collections import deque
import torch.optim as optim

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Using device: cpu


In [2]:
import json
# Parse schedule and map stations to
with open('EXP-TRAINS.json', 'r') as f:
    train_data = json.load(f) #[cite: 2]

G = nx.DiGraph()
node_map = {}
node_idx = 0

for train in train_data:
    route = train['trainRoute'] #[cite: 2]
    for i in range(len(route) - 1):
        src = route[i]['stationName'] #[cite: 2]
        dst = route[i+1]['stationName'] #[cite: 2]

        if src not in node_map:
            node_map[src] = node_idx
            node_idx += 1
        if dst not in node_map:
            node_map[dst] = node_idx
            node_idx += 1

        G.add_edge(node_map[src], node_map[dst])

num_nodes = len(node_map)
edges = list(G.edges())
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous().to(device)

# Generate temporal features (X_seq: 24h traffic) and target congestion (y)[cite: 1]
seq_length = 24
X_seq = torch.rand((seq_length, num_nodes, 3)).to(device) #[cite: 1]
y = torch.rand((num_nodes, 1)).to(device) #[cite: 1]

In [3]:
class TGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(TGCN, self).__init__()
        self.gcn = GCNConv(in_channels, hidden_channels) #[cite: 1]
        self.gru = torch.nn.GRUCell(hidden_channels, hidden_channels) #[cite: 1]
        self.linear = torch.nn.Linear(hidden_channels, 1) #[cite: 1]

    def forward(self, x_seq, edge_index):
        time_steps, num_nodes, _ = x_seq.size()
        hidden_state = torch.zeros(num_nodes, self.gru.hidden_size).to(device) #[cite: 1]

        for t in range(time_steps):
            gcn_out = F.relu(self.gcn(x_seq[t], edge_index)) #[cite: 1]
            hidden_state = self.gru(gcn_out, hidden_state) #[cite: 1]

        out = self.linear(hidden_state)
        return torch.sigmoid(out) #[cite: 1]

model = TGCN(in_channels=3, hidden_channels=32).to(device) #[cite: 1]
optimizer = torch.optim.Adam(model.parameters(), lr=0.01) #[cite: 1]
criterion = torch.nn.MSELoss() #[cite: 1]

In [7]:
epochs = 100
model.train() #[cite: 1]

for epoch in range(epochs):
    optimizer.zero_grad() #[cite: 1]
    predicted_congestion = model(X_seq, edge_index) #[cite: 1]
    loss = criterion(predicted_congestion, y) #[cite: 1]
    loss.backward() #[cite: 1]
    optimizer.step() #[cite: 1]

    if epoch % 20 == 0:
        print(f"Epoch {epoch} | T-GCN Loss: {loss.item():.4f}") #[cite: 1]

Epoch 0 | T-GCN Loss: 0.0831
Epoch 20 | T-GCN Loss: 0.0815
Epoch 40 | T-GCN Loss: 0.0784
Epoch 60 | T-GCN Loss: 0.0761
Epoch 80 | T-GCN Loss: 0.0746


In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Convert PyTorch tensors to NumPy arrays
# Ensure gradient tracking is turned off using detach()
y_true = y.detach().cpu().numpy()
y_pred = predicted_congestion.detach().cpu().numpy()

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("--- T-GCN Evaluation Metrics ---")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"RMSE (Root Mean Square Error): {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

# Note: Since the graph was populated with random mock data in Cell 2,
# these scores will look poor until trained on real dataset tensors.

--- T-GCN Evaluation Metrics ---
MAE (Mean Absolute Error): 0.2161
RMSE (Root Mean Square Error): 0.2583
R² Score: 0.2085


In [14]:
from google.colab import files
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# -------------------------------------------------------------
# 1 & 2. Graph Setup and T-GCN Cost Logic
# -------------------------------------------------------------
# (Assuming model and congestion_scores are already defined in your environment)
# For the sake of this isolated visualizer, we mock congestion_scores if undefined
if 'congestion_scores' not in locals():
    congestion_scores = np.random.rand(100)

G_sim = nx.Graph()
# Ensure fallback graph has a clear diamond structure for rerouting if G is empty
if len(G_sim.nodes()) < 6:
    edges = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5),  # Main line
             (1, 6), (6, 7), (7, 4)]                  # Bypass line
    G_sim.add_edges_from(edges)

CONGESTION_WEIGHT = 5.0

def update_edge_weights(graph, bottleneck_edges=None):
    if bottleneck_edges is None:
        bottleneck_edges = set()

    for u, v in graph.edges():
        u_score = float(congestion_scores[u]) if u < len(congestion_scores) else 0.1
        v_score = float(congestion_scores[v]) if v < len(congestion_scores) else 0.1
        avg_congestion = (u_score + v_score) / 2.0

        cost = 1.0 + (CONGESTION_WEIGHT * avg_congestion)
        if (u, v) in bottleneck_edges or (v, u) in bottleneck_edges:
            cost += 100.0  # Heavy penalty for occupied block

        graph[u][v]['weight'] = cost

# -------------------------------------------------------------
# 3. Simulate Two-Train Conflict and Dynamic Rerouting
# -------------------------------------------------------------
train1_start, train1_dest = 0, 5
train2_start, train2_dest = 1, 5

# Step A: Initial optimal routes without conflict
update_edge_weights(G_sim)
path1 = nx.shortest_path(G_sim, source=train1_start, target=train1_dest, weight='weight')
initial_path2 = nx.shortest_path(G_sim, source=train2_start, target=train2_dest, weight='weight')

# Identify bottleneck (first shared intermediate node)
shared_nodes = [node for node in path1[1:-1] if node in initial_path2[1:-1]]
conflict_node = shared_nodes[0] if shared_nodes else path1[len(path1)//2]

# Step B: Train 1 claims the block; update costs
occupied_edges = {(path1[i], path1[i+1]) for i in range(len(path1)-1)}
update_edge_weights(G_sim, bottleneck_edges=occupied_edges)

# Step C: Dynamic Rerouting of Train 2
try:
    rerouted_path2 = nx.shortest_path(G_sim, source=train2_start, target=train2_dest, weight='weight')
except nx.NetworkXNoPath:
    rerouted_path2 = initial_path2

print(f"Train 1 Route (Green): {path1}")
print(f"Train 2 Original Route (Red Dashed): {initial_path2}")
print(f"Train 2 Rerouted Path (Blue): {rerouted_path2}")

# -------------------------------------------------------------
# 4. Multi-Agent Matplotlib Animation Simulation (SMOOTH & LONG)
# -------------------------------------------------------------
pos = nx.spring_layout(G_sim, seed=42)

# --- Interpolation Logic for Smooth Animation ---
STEPS_PER_EDGE = 15  # Increase this to make the animation slower/longer

def generate_smooth_trajectory(path, positions, steps):
    trajectory = []
    for i in range(len(path) - 1):
        start_pos = np.array(positions[path[i]])
        end_pos = np.array(positions[path[i+1]])
        for step in range(steps):
            alpha = step / steps
            interp_pos = start_pos * (1 - alpha) + end_pos * alpha
            trajectory.append(interp_pos)
    trajectory.append(np.array(positions[path[-1]]))
    return trajectory

t1_points = generate_smooth_trajectory(path1, pos, STEPS_PER_EDGE)
t2_points = generate_smooth_trajectory(rerouted_path2, pos, STEPS_PER_EDGE)

# Pad the shorter trajectory so they finish together
max_frames = max(len(t1_points), len(t2_points))
t1_points += [t1_points[-1]] * (max_frames - len(t1_points))
t2_points += [t2_points[-1]] * (max_frames - len(t2_points))

# --- Plotting Setup ---
fig, ax = plt.subplots(figsize=(16, 10))

def animate_simulation(frame):
    ax.clear()

    # 1. Base infrastructure
    nx.draw_networkx_nodes(G_sim, pos, ax=ax, node_color='#EEEEEE', node_size=300)
    nx.draw_networkx_edges(G_sim, pos, ax=ax, edge_color='#CCCCCC', width=1.0)

    # 2. Highlight Paths
    # Original conflicting path for Train 2 (Dashed Red)
    orig_t2_edges = list(zip(initial_path2, initial_path2[1:]))
    nx.draw_networkx_edges(G_sim, pos, edgelist=orig_t2_edges, edge_color='#E74C3C', width=2.5, style='dashed', ax=ax, label='Train 2 Original Route (Blocked)')

    # Train 1 Path (Solid Green)
    t1_edges = list(zip(path1, path1[1:]))
    nx.draw_networkx_edges(G_sim, pos, edgelist=t1_edges, edge_color='#2ECC71', width=3.5, ax=ax, label='Train 1 Priority Route')

    # Train 2 Rerouted Path (Solid Blue)
    t2_edges = list(zip(rerouted_path2, rerouted_path2[1:]))
    nx.draw_networkx_edges(G_sim, pos, edgelist=t2_edges, edge_color='#3498DB', width=3.5, ax=ax, label='Train 2 Rerouted Path')

    # 3. Highlight Conflict Node
    if conflict_node in G_sim:
        nx.draw_networkx_nodes(G_sim, pos, nodelist=[conflict_node], node_color='#E67E22',
                               node_size=600, ax=ax)
        # Add a text alert near the bottleneck
        cx, cy = pos[conflict_node]
        ax.text(cx, cy + 0.08, "⚠ CONGESTED BLOCK\nT-GCN Penalty Applied",
                color='#C0392B', fontsize=10, fontweight='bold', ha='center',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.2'))

    # 4. Draw Moving Trains (Interpolated positions)
    curr_t1_pos = t1_points[frame]
    curr_t2_pos = t2_points[frame]

    ax.scatter(curr_t1_pos[0], curr_t1_pos[1], s=400, c='#27AE60', zorder=5, edgecolors='black')
    ax.scatter(curr_t2_pos[0], curr_t2_pos[1], s=400, c='#2980B9', zorder=5, edgecolors='black')

    # Train icons/labels
    ax.text(curr_t1_pos[0], curr_t1_pos[1], 'T1', color='white', fontweight='bold', ha='center', va='center', zorder=6)
    ax.text(curr_t2_pos[0], curr_t2_pos[1], 'T2', color='white', fontweight='bold', ha='center', va='center', zorder=6)

    # 5. Station Labels
    nx.draw_networkx_labels(G_sim, pos, font_size=10, font_family='sans-serif', ax=ax)

    # 6. Status and Legends
    progress_pct = int((frame / max_frames) * 100)
    ax.set_title(
        f"Dynamic Conflict Resolution Simulation | Progress: {progress_pct}%\n"
        f"Train 2 avoids congested Node {conflict_node} via Edge Connectivity Rerouting",
        fontsize=14, fontweight='bold', pad=15
    )

    # Custom legend to map colors properly
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(loc='upper right', framealpha=0.9, fontsize=10)
    ax.axis('off')

ani = animation.FuncAnimation(fig, animate_simulation, frames=max_frames, interval=50, repeat=True)
plt.close()

HTML(ani.to_jshtml())


# Define the writer
writer = animation.writers['ffmpeg'](fps=30, metadata=dict(artist='Me'), bitrate=1800)

# Save the animation
ani.save('my_animation.mp4', writer=writer)

# Download the file
files.download('my_animation.mp4')

Train 1 Route (Green): [0, 1, 6, 7, 4, 5]
Train 2 Original Route (Red Dashed): [1, 6, 7, 4, 5]
Train 2 Rerouted Path (Blue): [1, 2, 3, 4, 5]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>